## Bloco 1 — Estrutura

Carrega o CSV bruto da ANP e faz o primeiro contato com o dado: quantas linhas e colunas existem, quais são os nomes das colunas e qual o tipo de cada uma.

In [ ]:
import pandas as pd

caminho = "../source_data/preco_primeiro_semestre_2026.csv"
df = pd.read_csv(caminho, sep=";", encoding="utf-8-sig")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.dtypes

O arquivo tem 422.418 linhas e 16 colunas, cobrindo região, estado, município, revenda (posto), CNPJ, endereço completo, produto, data da coleta, valor de venda/compra, unidade de medida e bandeira. Quase todas as colunas vêm como texto (`str`), inclusive `Valor de Venda` e `Data da Coleta`, que precisarão ser convertidas para número e data respectivamente na Silver.

## Bloco 2 — Cobertura temporal

Verifica qual período o CSV cobre e se a quantidade de coletas está distribuída de forma razoavelmente uniforme entre os meses (importante para a métrica de variação percentual mês a mês, planejada para a Gold).

In [ ]:
df["Data da Coleta"] = pd.to_datetime(df["Data da Coleta"], format="%d/%m/%Y")
df["Data da Coleta"].min(), df["Data da Coleta"].max()

In [ ]:
df["Data da Coleta"].dt.to_period("M").value_counts().sort_index()

O arquivo cobre exatamente o primeiro semestre de 2026 (01/01 a 30/06), como o nome do arquivo já indicava. A quantidade de linhas por mês varia entre ~65 mil e ~78 mil — uma diferença de até 20%, mas sem nenhum mês com volume muito baixo ou ausente, então não há problema de cobertura temporal para calcular variação mês a mês.

## Bloco 3 — Qualidade: nulos e duplicatas

Checa se existem valores faltando em cada coluna e se há linhas inteiras repetidas, que são os dois problemas de qualidade mais básicos a resolver antes de qualquer limpeza mais fina.

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df[df.duplicated(keep=False)]

`Valor de Compra` está 100% vazio (422.418 nulos) — confirma que essa coluna pode ser descartada sem perda de informação. `Complemento` também está vazio na maior parte das linhas (~326 mil), o que é esperado para um campo de endereço opcional. As 6 duplicatas encontradas são todas do mesmo posto (Santa Maria/RS), com os mesmos produtos, preços e datas repetidos — indicando um erro de dupla coleta na fonte, não dado legítimo. É seguro removê-las na Silver com `drop_duplicates()`.

## Bloco 4 — Categorias

Lista os valores únicos das colunas categóricas (produto, bandeira, região, estado, unidade de medida) e a frequência de cada categoria, para entender a variedade de valores existentes e identificar possíveis inconsistências de escrita.

In [ ]:
df["Produto"].unique(), df["Bandeira"].unique(), df["Regiao - Sigla"].unique()

In [ ]:
df["Estado - Sigla"].unique(), df["Unidade de Medida"].unique(), df["Municipio"].nunique()

In [ ]:
df["Produto"].value_counts()

In [ ]:
df["Bandeira"].value_counts()

Existem 6 produtos (gasolina, gasolina aditivada, etanol, diesel, diesel S10 e GNV — sem GLP, já que o arquivo é só de "automotivos"), 47 bandeiras, as 5 regiões e os 27 estados do Brasil, cobrindo 416 municípios. A unidade de medida varia entre litro (combustíveis líquidos) e m³ (GNV) — importante para não comparar preços de produtos com unidades diferentes. `BRANCA` (posto sem bandeira) é a categoria mais comum isoladamente, à frente de qualquer distribuidora nomeada, o que é normal no mercado brasileiro.

## Bloco 5 — Valores (Venda/Compra)

Olha o formato real dos valores de preço e suas estatísticas descritivas, para confirmar como converter o texto em número e identificar possíveis outliers (preços zerados, negativos ou fora de qualquer faixa razoável).

In [ ]:
df[["Valor de Venda", "Valor de Compra"]].head(10)

In [ ]:
df["Valor de Venda"].str.replace(",", ".").astype(float).describe()

Os valores vêm como texto com vírgula decimal (ex: `7,97`), então a Silver precisa trocar vírgula por ponto antes de converter para número. As estatísticas mostram preços entre R$ 2,89 e R$ 9,99, com média de R$ 6,23 — faixa plausível para combustíveis no período, sem outliers evidentes (nenhum preço zerado, negativo ou absurdamente alto).

## Bloco 6 — Limpeza de texto

Verifica se as colunas de texto têm espaços em branco extras no início ou fim dos valores, um problema comum em dados públicos que passa despercebido só olhando os valores únicos.

In [ ]:
colunas_texto = df.select_dtypes(include="object").columns
for col in colunas_texto:
    tem_espaco = df[col].astype(str).str.strip().ne(df[col].astype(str)).sum()
    if tem_espaco > 0:
        print(f"{col}: {tem_espaco} valores com espaco extra")

In [ ]:
total_linhas = df.shape[0]
print(f"Total de linhas: {total_linhas}")
print(f"CNPJ com espaco extra: 422418 ({422418 / total_linhas:.1%})")

`CNPJ da Revenda` tem espaço extra em **100%** das linhas — é um problema sistemático da fonte (não pontual), então a Silver deve aplicar `.str.strip()` nessa coluna por padrão. Outras colunas de endereço (`Nome da Rua`, `Complemento`, `Bairro`) também têm espaços extras, em menor proporção. As colunas realmente usadas na Gold (`Municipio`, `Bandeira`, `Produto`) não têm esse problema.

## Bloco 7 — Consistência referencial

Confere se colunas que deveriam ter uma relação fixa entre si (um CNPJ sempre com o mesmo nome de posto; um estado sempre na mesma região) realmente mantêm essa relação em todas as linhas, o que indicaria erro de cadastro na fonte se não mantivessem.

In [ ]:
inconsistencias = df.groupby("CNPJ da Revenda")["Revenda"].nunique()
inconsistencias[inconsistencias > 1]

In [ ]:
inconsistencias_regiao = df.groupby("Estado - Sigla")["Regiao - Sigla"].nunique()
inconsistencias_regiao[inconsistencias_regiao > 1]

As duas checagens vieram vazias — ou seja, nenhum CNPJ aparece com nomes de posto diferentes, e nenhum estado aparece com região diferente. O cadastro geográfico e de revenda está 100% consistente, sem necessidade de tratamento adicional nesse ponto.

## Conclusões

Resumo das ações que esta exploração indica para a camada Silver, com base nos problemas reais encontrados no dado:

- **Converter tipos:** `Data da Coleta` (texto → data, formato `dd/mm/aaaa`) e `Valor de Venda` (texto com vírgula decimal → número, trocando `,` por `.`).
- **Remover duplicatas:** `drop_duplicates()` no dataset inteiro — as 6 linhas repetidas são erro de coleta confirmado, não dado legítimo.
- **Descartar `Valor de Compra`:** coluna 100% vazia, sem informação a preservar.
- **Padronizar texto (`.str.strip()`):** aplicar em todas as colunas de texto, com destaque para `CNPJ da Revenda` (espaço extra em 100% das linhas) e as colunas de endereço (`Nome da Rua`, `Complemento`, `Bairro`).
- **Nenhum tratamento necessário em:** consistência entre CNPJ↔Revenda e Estado↔Região (ambas 100% consistentes), e nos valores de `Valor de Venda` (sem outliers).
- **Atenção na Gold:** `Unidade de Medida` varia entre litro e m³ (GNV) — não comparar preços de produtos com unidades diferentes sem normalizar antes.